# 02 · LangChain Tool-Calling Agent

The **agent** layer converts the raw LLM into a *decision-maker*: it chooses which tool to invoke based
on the user's natural-language intent, runs the tool, inspects the result, and replies.

## Architecture

```
User message
     │
     ▼
[ RAG context inject ]  ←── ChromaDB retriever
     │
     ▼
[ System prompt + conversation history + user message ]
     │
     ▼
  Groq LLM  (meta-llama/llama-4-scout-17b-16e-instruct)
     │
     ├── tool_calls? ──► ToolExecutor ──► tool result ──► back to LLM
     │                   (max 4 iterations)
     │
     └── AIMessage  ──► response streamed to frontend
```

## Available Tools

| Tool | Trigger intent | Outcome |
|---|---|---|
| `search_portfolio` | "show me your work", "case studies" | Returns portfolio markdown |
| `book_consultation` | "book a call", "schedule a meeting" | Returns consultation link |
| `start_ux_audit` | "audit my website" | Redirects frontend to UX Audit panel |
| `start_data_insights` | "analyse my data", "upload csv" | Redirects frontend to Data Insights panel |
| `start_project_scope` | "scope a project", "I have an idea" | Redirects frontend to Scope panel |
| `create_user_persona` | "create a persona", "define my audience" | Redirects frontend to Persona panel |


In [1]:
# %pip install -q langchain langchain-groq langchain-core python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os, json, asyncio
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(dotenv_path=Path('..') / '.env')

GROQ_API_KEY = os.environ['GROQ_API_KEY']
print('Groq key loaded:', GROQ_API_KEY[:8] + '…')

Groq key loaded: gsk_xuu0…


## Define the Tools

In [3]:
from langchain_core.tools import tool

@tool
def search_portfolio(query: str) -> str:
    """Search BrixoAI's portfolio and case studies. Use when user asks about past work or projects."""
    return (
        '## BrixoAI Portfolio\n\n'
        '- **Mystika** — immersive astrology lifestyle app\n'
        '- **PetShop Blue** — e-commerce for pet accessories\n'
        'You can view detailed case studies at /case-studies'
    )

@tool
def book_consultation(reason: str = '') -> str:
    """Book a discovery consultation with BrixoAI. Use when the user wants to schedule a meeting or discuss a project."""
    return 'https://calendly.com/brixoai/discovery'

@tool
def start_ux_audit(url: str = '') -> str:
    """Start a UX audit for a website URL or uploaded screenshot. Triggers the UX Audit panel."""
    return '__TRIGGER:ux_audit__'

@tool
def start_data_insights(topic: str = '') -> str:
    """Open the Data Insights panel for CSV/Excel analysis. Use when user asks to analyse data."""
    return '__TRIGGER:data_insights__'

@tool
def start_project_scope(description: str = '') -> str:
    """Open the Project Scope panel. Use when the user wants to scope a new project or discuss their idea."""
    return '__TRIGGER:scope_project__'

@tool
def create_user_persona(audience: str = '') -> str:
    """Open the User Persona generator. Use when user wants to define their target audience."""
    return '__TRIGGER:user_persona__'

tools = [
    search_portfolio,
    book_consultation,
    start_ux_audit,
    start_data_insights,
    start_project_scope,
    create_user_persona,
]

print('Registered tools:', [t.name for t in tools])

Registered tools: ['search_portfolio', 'book_consultation', 'start_ux_audit', 'start_data_insights', 'start_project_scope', 'create_user_persona']


## Build the Agent

In [10]:
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage

SYSTEM_PROMPT = """
You are BrixoAI Assistant, a helpful AI consultant for BrixoAI design studio.
You answer questions about AI, UX design, data analytics, and BrixoAI services.
When appropriate, use the tools available to you rather than answering from memory alone.
If a user wants to analyse data, audit a website, scope a project, or create a user persona,
always invoke the corresponding tool — do not explain how to do it manually.
""".strip()

llm = ChatGroq(model='meta-llama/llama-4-scout-17b-16e-instruct', temperature=0.4)
llm_with_tools = llm.bind_tools(tools)

tool_map = {t.name: t for t in tools}

print('LLM with tools ready')


LLM with tools ready


## Agentic Loop

The loop continues until the model produces an `AIMessage` with no `tool_calls`, or until `MAX_ITER`.

In [11]:
async def run_agent(user_message: str, max_iter: int = 4, verbose: bool = True) -> str:
    """Run the agentic loop. Returns the final text response."""
    messages = [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=user_message),
    ]

    for iteration in range(max_iter):
        if verbose:
            print(f'\n[Iteration {iteration + 1}]')

        response: AIMessage = await llm_with_tools.ainvoke(messages)

        if not response.tool_calls:
            if verbose:
                print('  → Final response (no tool calls)')
            return response.content

        # Execute each tool call
        for tc in response.tool_calls:
            tool_name = tc['name']
            tool_args = tc['args']
            if verbose:
                print(f'  → Calling tool: {tool_name}({tool_args})')

            tool_fn = tool_map[tool_name]
            tool_result = tool_fn.invoke(tool_args)
            if verbose:
                print(f'     Result: {tool_result}')

            messages += [
                response,
                ToolMessage(content=str(tool_result), tool_call_id=tc['id']),
            ]

    return response.content or '(no response)'

## Demo Conversations

In [12]:
response = await run_agent('Can you show me some of your past work?')
print('\nFinal Response:\n', response)


[Iteration 1]
  → Calling tool: search_portfolio({'query': 'past work'})
     Result: ## BrixoAI Portfolio

- **Mystika** — immersive astrology lifestyle app
- **PetShop Blue** — e-commerce for pet accessories
You can view detailed case studies at /case-studies

[Iteration 2]
  → Final response (no tool calls)

Final Response:
 You can view detailed case studies on our website. Would you like to know more about a specific project?


In [13]:
response = await run_agent('I want to audit my website at https://example.com')
print('\nFinal Response:\n', response)


[Iteration 1]
  → Calling tool: start_ux_audit({'url': 'https://example.com'})
     Result: __TRIGGER:ux_audit__

[Iteration 2]
  → Calling tool: start_ux_audit({'url': 'https://example.com'})
     Result: __TRIGGER:ux_audit__

[Iteration 3]
  → Calling tool: start_ux_audit({'url': 'https://example.com'})
     Result: __TRIGGER:ux_audit__

[Iteration 4]
  → Calling tool: start_ux_audit({'url': 'https://example.com'})
     Result: __TRIGGER:ux_audit__

Final Response:
 I can't provide direct feedback on your website, but I can guide you through the UX audit process. 

To proceed, please use the following tool: 




In [14]:
response = await run_agent('I have sales data I want to analyse — can you help?')
print('\nFinal Response:\n', response)


[Iteration 1]
  → Calling tool: start_data_insights({'topic': 'sales data analysis'})
     Result: __TRIGGER:data_insights__

[Iteration 2]
  → Final response (no tool calls)

Final Response:
 I can help you analyze your sales data. I will open the Data Insights panel for CSV/Excel analysis. Please provide the data or upload your file.


In [15]:
# General question — should NOT invoke any tool
response = await run_agent('What is the difference between UI and UX?')
print('\nFinal Response:\n', response)


[Iteration 1]
  → Final response (no tool calls)

Final Response:
 UI (User Interface) and UX (User Experience) are two related but distinct concepts in the field of design and human-computer interaction.

UI refers to the visual elements and interactions of a product, such as buttons, typography, color schemes, and layouts. It encompasses the tangible aspects of a product that users interact with, like the interface of a website, app, or software. The goal of UI design is to create an intuitive and aesthetically pleasing interface that allows users to interact with the product efficiently.

UX, on the other hand, encompasses the entire experience a user has with a product, from the initial interaction to the final outcome. It involves understanding user needs, behaviors, and motivations to create a product that provides value and satisfaction. UX design considers the user's journey, including their emotions, perceptions, and pain points, to craft an experience that is seamless, intui

## Inspecting Tool Binding

We can inspect the JSON schema that gets sent to the LLM.

In [16]:
for t in tools:
    schema = t.args_schema.schema() if t.args_schema else {}
    print(f'Tool: {t.name}')
    print(f'  Description: {t.description[:80]}')
    props = schema.get('properties', {})
    if props:
        for k, v in props.items():
            print(f'  Param: {k} ({v.get("type", "any")})')
    print()

Tool: search_portfolio
  Description: Search BrixoAI's portfolio and case studies. Use when user asks about past work 
  Param: query (string)

Tool: book_consultation
  Description: Book a discovery consultation with BrixoAI. Use when the user wants to schedule 
  Param: reason (string)

Tool: start_ux_audit
  Description: Start a UX audit for a website URL or uploaded screenshot. Triggers the UX Audit
  Param: url (string)

Tool: start_data_insights
  Description: Open the Data Insights panel for CSV/Excel analysis. Use when user asks to analy
  Param: topic (string)

Tool: start_project_scope
  Description: Open the Project Scope panel. Use when the user wants to scope a new project or 
  Param: description (string)

Tool: create_user_persona
  Description: Open the User Persona generator. Use when user wants to define their target audi
  Param: audience (string)



C:\Users\User\AppData\Local\Temp\ipykernel_16592\758799259.py:2: PydanticDeprecatedSince20: The `schema` method is deprecated; use `model_json_schema` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  schema = t.args_schema.schema() if t.args_schema else {}


## Summary

- The agent uses **OpenAI-compatible tool-calling** via Groq — the model outputs structured JSON rather than parsing special tokens.
- The agentic loop runs at most **4 iterations** to avoid runaway chains.
- Terminal tools (UX audit, scope, etc.) return `__TRIGGER:*__` tokens which the frontend interprets as panel redirects.
- In production, LangSmith traces every iteration for debugging and evaluation.